# reflection

In [ ]:
///- --test static

open testing

In [ ]:
open sm'_operators

## reflection

### get_union_fields

In [ ]:
inl get_union_fields forall union_type. () : list (string * union_type) =
    real
        real_core.union_to_record
            `union_type
            forall union_record_type. =>
                real_core.record_type_fold
                    fun acc key =>
                        forall value. =>
                            inl value =
                                typecase value with
                                | () => $'' : value
                                | _ =>
                                    backend_switch `value `({}) {
                                        Gleam =
                                            (fun () =>
                                                $'Nil' : value
                                            ) : () -> value
                                        Lua =
                                            (fun () =>
                                                $'nil' : value
                                            ) : () -> value
                                        Fsharp =
                                            (fun () =>
                                                $'Unchecked.defaultof<_>' : value
                                            ) : () -> value
                                        Python =
                                            (fun () =>
                                                inl is_python =
                                                    backend_switch `bool `({}) {
                                                        Gleam = (fun () => false) : () -> bool
                                                        Lua = (fun () => false) : () -> bool
                                                        Fsharp = (fun () => false) : () -> bool
                                                        Python = (fun () => true) : () -> bool
                                                    }
                                                if is_python
                                                then
                                                    typecase value with
                                                    | i8 => 0i8
                                                    | i16 => 0i16
                                                    | i32 => 0i32
                                                    | i64 => 0i64
                                                    | u8 => 0u8
                                                    | u16 => 0u16
                                                    | u32 => 0u32
                                                    | u64 => 0u64
                                                    | f32 => 0.0f32
                                                    | f64 => 0.0f64
                                                    | bool => false
                                                    | char => '\0'
                                                    | _ => $'None' : value
                                                else $'None' : value
                                            ) : () -> value
                                        Rust =
                                            (fun () =>
                                                $'Default::default()' : value
                                            ) : () -> value
                                        Zig =
                                            (fun () =>
                                                $'std.mem.zeroes(`value)' : value
                                            ) : () -> value
                                        TypeScript =
                                            (fun () =>
                                                typecase value with
                                                | i8 => 0i8
                                                | i16 => 0i16
                                                | i32 => 0i32
                                                | i64 => 0i64
                                                | u8 => 0u8
                                                | u16 => 0u16
                                                | u32 => 0u32
                                                | u64 => 0u64
                                                | f32 => 0.0f32
                                                | f64 => 0.0f64
                                                | bool => false
                                                | char => '\0'
                                                | _ => $'undefined' : value
                                            ) : () -> value
                                    }
                            inl item = real_core.nominal_create `union_type (key, value)
                            inl key' = sm'_real.symbol_to_string_real `(`key)
                            (::) `(string * union_type) (key', item) acc
                    (Nil `(string * union_type))
                    `union_record_type
union reflection_test_target = Wasm | Contract

In [1]:
///- --test
///> fsharp
///> rust
///> typescript
///> cuda

get_union_fields ()
|> listm'.box
|> listm'.to_array'
|> a
|> am'.sort_by snd
|> fun (a x : _ int _) => x
|> _assert_eq' ;[ "Wasm", Wasm; "Contract", Contract ]

get_union_fields ()
|> listm'.box
|> listm'.to_array'
|> a
|> am'.sort_by snd
|> fun (a x : _ int _) => x
|> _assert_eq' ;[ "Some", Some 0i32; "None", None ]

.fsx (fsharp)
__assert_eq' { expected = [(Wasm, Wasm), (Contract, Contract)] }
__assert_eq' { expected = [(Some, Some(0)), (None, None)] }
.rs (rust)
__assert_eq' { expected = [(Wasm, Wasm), (Contract, Contract)] }
__assert_eq' { expected = [(Some, Some(0)), (None, None)] }
.ts (typescript)
__assert_eq' { expected = [(Wasm, Wasm), (Contract, Contract)] }
__assert_eq' { expected = [(Some, Some(0)), (None, None)] }
.py (cuda)
__assert_eq' { expected = [(Wasm, Wasm), (Contract, Contract)] }
__assert_eq' { expected = [(Some, Some(0)), (None, None)] }

0

### get_union_fields_untag

In [ ]:
inl get_union_fields_untag forall union_type. () : list (string * union_type) =
    real
        real_core.union_to_record
            `union_type
            forall union_record_type. =>
                inl result =
                    real_core.record_type_fold_back
                        fun _key =>
                            forall value. (acc, (i : i32)) =>
                                inl key, item : (string * union_type) =
                                    real_core.union_untag `union_type i
                                        (fun key => forall value. =>
                                            inl key' = sm'_real.symbol_to_string_real `(`key)
                                            inl value =
                                                typecase value with
                                                | () => $'' : value
                                                | _ =>
                                                    backend_switch `value `({}) {
                                                        Gleam =
                                                            (fun () =>
                                                                $'Nil' : value
                                                            ) : () -> value
                                                        Lua =
                                                            (fun () =>
                                                                $'nil' : value
                                                            ) : () -> value
                                                        Fsharp =
                                                            (fun () =>
                                                                $'Unchecked.defaultof<_>' : value
                                                            ) : () -> value
                                                        Python =
                                                            (fun () =>
                                                                $'None' : value
                                                            ) : () -> value
                                                        Rust =
                                                            (fun () =>
                                                                $'Default::default()' : value
                                                            ) : () -> value
                                                        TypeScript =
                                                            (fun () =>
                                                                $'undefined' : value
                                                            ) : () -> value
                                                        Zig =
                                                            (fun () =>
                                                                $'std.mem.zeroes(`value)' : value
                                                            ) : () -> value
                                                    }
                                            inl item = real_core.nominal_create `union_type (key, value)
                                            key', item
                                        )
                                        (fun _ =>
                                            failwith
                                                `(string * union_type)
                                                "reflection.get_union_fields_untag / invalid tag"
                                        )
                                (::) `(string * union_type) (key, item) acc, (+) `i32 i 1
                        `union_record_type
                        (Nil `(string * union_type), 0i32)
                inl result = fst `(list (string * union_type)) `i32 result
                listm.rev `(string * union_type) result

In [2]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust
///> typescript

get_union_fields_untag ()
|> _assert_eq' [ "Wasm", Wasm; "Contract", Contract ]

get_union_fields_untag ()
|> _assert_eq' [ "Some", Some (); "None", None ]

.gleam (gleam)
__assert_eq' { expected = Cons((Wasm, Wasm), Cons((Contract, Contract), Nil)) }
__assert_eq' { expected = Cons((Some, Some), Cons((None, None), Nil)) }
.lua (lua)
__assert_eq' { expected = Cons((Wasm, Wasm), Cons((Contract, Contract), Nil)) }
__assert_eq' { expected = Cons((Some, Some), Cons((None, None), Nil)) }
.fsx (fsharp)
__assert_eq' { expected = Cons((Wasm, Wasm), Cons((Contract, Contract), Nil)) }
__assert_eq' { expected = Cons((Some, Some), Cons((None, None), Nil)) }
.py (cuda)
__assert_eq' { expected = Cons((Wasm, Wasm), Cons((Contract, Contract), Nil)) }
__assert_eq' { expected = Cons((Some, Some), Cons((None, None), Nil)) }
.rs (rust)
__assert_eq' { expected = Cons((Wasm, Wasm), Cons((Contract, Contract), Nil)) }
__assert_eq' { expected = Cons((Some, Some), Cons((None, None), Nil)) }
.ts (typescript)
__assert_eq' { expected = Cons((Wasm, Wasm), Cons((Contract, Contract), Nil)) }
__assert_eq' { expected = Cons((Some, Some), Cons((None, None), Nil)) }

0

### union_try_pick

In [ ]:
inl union_try_pick forall t. (key : string) : option t =
    inl fields = real get_union_fields_untag `t ()
    fields
    ++ (
        fields
        |> listm.map fun key', x =>
            key' |> sm'.to_lower, x
    )
    |> listm'.box
    |> listm'.to_array'
    |> fun x => a x : _ int _
    |> am'.try_pick fun key', x =>
        if key' =. key
        then Some x
        else None

In [3]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust
///> typescript

"wasm"
|> union_try_pick
|> _assert_eq (Some Wasm)

"Wasm"
|> union_try_pick
|> _assert_eq (Some Wasm)

.gleam (gleam)
__assert_eq { expected = Some(Wasm) }
__assert_eq { expected = Some(Wasm) }
.lua (lua)
__assert_eq { expected = Some(Wasm) }
__assert_eq { expected = Some(Wasm) }
.fsx (fsharp)
__assert_eq { expected = Some(Wasm) }
__assert_eq { expected = Some(Wasm) }
.py (cuda)
__assert_eq { expected = Some(Wasm) }
__assert_eq { expected = Some(Wasm) }
.rs (rust)
__assert_eq { expected = Some(Wasm) }
__assert_eq { expected = Some(Wasm) }
.ts (typescript)
__assert_eq { expected = Some(Wasm) }
__assert_eq { expected = Some(Wasm) }

0

### union_try_pick'

In [ ]:
inl union_try_pick' forall t. (key : string) : option t =
    inl fields = real get_union_fields_untag `t ()
    fields
    ++ (
        fields
        |> listm.map fun key', x =>
            key' |> sm'.to_lower, x
    )
    |> listm'.try_pick fun key', x =>
        if key' =. key
        then x |> Some
        else None

In [4]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust
///> typescript

"wasm"
|> union_try_pick'
|> _assert_eq (Some Wasm)

"Wasm"
|> union_try_pick'
|> _assert_eq (Some Wasm)

.gleam (gleam)
__assert_eq { expected = Some(Wasm) }
__assert_eq { expected = Some(Wasm) }
.lua (lua)
__assert_eq { expected = Some(Wasm) }
__assert_eq { expected = Some(Wasm) }
.fsx (fsharp)
__assert_eq { expected = Some(Wasm) }
__assert_eq { expected = Some(Wasm) }
.py (cuda)
__assert_eq { expected = Some(Wasm) }
__assert_eq { expected = Some(Wasm) }
.rs (rust)
__assert_eq { expected = Some(Wasm) }
__assert_eq { expected = Some(Wasm) }
.ts (typescript)
__assert_eq { expected = Some(Wasm) }
__assert_eq { expected = Some(Wasm) }

0

### union_cases_fold

In [ ]:
inl union_cases_fold forall union_type state. (step : state -> union_type -> state) (initial : state) : state =
    real
        real_core.union_to_record `union_type forall union_record_type. =>
            real_core.record_type_fold
                fun acc key => forall value. => step acc (real_core.nominal_create `union_type (key, ()))
                initial
                `union_record_type

In [5]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust
///> typescript

union_cases_fold (fun count (_ : reflection_test_target) => count + 1i32) 0i32
|> _assert_eq 2i32

union_cases_fold (fun found (target : reflection_test_target) => if target = Contract then true else found) false
|> _assert_eq true

.gleam (gleam)
__assert_eq { expected = 2 }
__assert_eq { expected = true }
.lua (lua)
__assert_eq { expected = 2 }
__assert_eq { expected = true }
.fsx (fsharp)
__assert_eq { expected = 2 }
__assert_eq { expected = true }
.py (cuda)
__assert_eq { expected = 2 }
__assert_eq { expected = true }
.rs (rust)
__assert_eq { expected = 2 }
__assert_eq { expected = true }
.ts (typescript)
__assert_eq { expected = 2 }
__assert_eq { expected = true }

0

### union_to_string

In [ ]:
inl union_to_string forall t. (x : t) : string =
    real get_union_fields_untag `t ()
    |> listm'.try_pick fun key, x' =>
        if x' = x
        then Some key
        else
            inl has_case =
                real
                    real_core.union_to_record
                        `t
                        forall union_record_type. =>
                            real_core.record_type_fold_back
                                fun _key =>
                                    forall value. acc =>
                                        if acc
                                        then acc
                                        else
                                            typecase value with
                                            | () => false
                                            | _ => true
                                `union_record_type
                                false
            if has_case |> not
            then None
            else
                inl x' = #?x' |> sm'.split "(" |> am'.index_base 0

                if #?x |> sm'.starts_with x'
                then Some key
                else None
    |> optionm.value

In [6]:
///- --test
///> fsharp
///> cuda
///> rust
///> typescript

Some true
|> union_to_string
|> _assert_eq' "Some"

.fsx (fsharp)
__assert_eq' { expected = Some }
.py (cuda)
__assert_eq' { expected = Some }
.rs (rust)
__assert_eq' { expected = Some }
.ts (typescript)
__assert_eq' { expected = Some }

0

### nameof

In [ ]:
inl nameof forall t. (x : t) : string =
    real
        real_core.record_type_fold_back
            fun key =>
                forall value. _ =>
                    sm'_real.symbol_to_string_real `(`key)
            `t
            ""

In [7]:
///- --test

{ test1 = ""; test2 = "" }
|> nameof
|> _assert_eq' "test1"

__assert_eq' { expected = test1 }

### get_record_fields

In [ ]:
inl get_record_fields forall t u. (x : t) : list (string * u) =
    real
        real_core.record_type_fold_back
            fun key =>
                forall u'. acc =>
                    inl k = sm'_real.symbol_to_string_real `(`key)
                    inl v = x key
                    (::) `(string * u') (k, v) acc
            `t
            (Nil `(string * u))

In [8]:
///- --test

{ a = "1"; b = "2" }
|> get_record_fields
|> _assert_eq' [ "a", "1"; "b", "2" ]

__assert_eq' { expected = Cons((a, 1), Cons((b, 2), Nil)) }

### get_functions_types

In [9]:
inl get_functions_types forall t {record}. (fns : t) =
    real
        inl get_function_type forall t. =
            inl args forall t {record}. : list (string * string) =
                real_core.record_type_fold_back
                    fun key =>
                        forall v. acc =>
                            inl k = sm'_real.symbol_to_string_real `(`key)
                            inl v = $'"`v"' : string
                            (::) `(string * string) (k, v) acc
                    `t
                    (Nil `(string * string))

            typecase t with
            | ~t -> ~u => args `t, ($'"`u"' : string)

        real_core.record_type_fold_back
            fun key =>
                forall v. acc =>
                    inl k = sm'_real.symbol_to_string_real `(`key)
                    inl args, result = get_function_type `v
                    (::) `(string * (list (string * string) * string)) (k, (args, result)) acc
            `(`fns)
            (Nil `(string * (list (string * string) * string)))
    |> fun x => x : list (string * (list (string * string) * string))

In [10]:
///- --test
///> fsharp

inl one ({ a } : { a : i32 }) : i32 = a + 1
inl two ({ a b } : { a : i32; b : i32 }) : i32 = a + b + 2
inl fns = { one two }

fns
|> get_functions_types
|> listm.map fun (name, args, result) => name, (args |> listm'.box |> listm'.to_array', result)
|> listm'.box
|> listm'.to_array'
|> sm'.format
|> _assert_eq' (
    [
        "one", ["a", "int32"], "int32"
        "two", ["a", "int32"; "b", "int32"], "int32"
    ]
    |> listm.map fun (name, args, result) => name, (args |> listm'.box |> listm'.to_array', result)
    |> listm'.box
    |> listm'.to_array'
    |> sm'.format
)

__assert_eq' { expected = [(one, [(a, int32)], int32), (two, [(a, int32), (b, int32)], int32)] }